# Chương 9. Tổng hợp dữ liệu theo nhóm

**Câu hỏi mở đầu:** Làm thế nào chuyển hàng nghìn bản ghi thành thông tin có ý nghĩa ở cấp nhóm?

Notebook này là tài nguyên đồng hành của chương. Mỗi phần đều đi theo nhịp **câu hỏi → dữ liệu → mã → kết quả → diễn giải → kiểm tra bằng chứng**.

## Mục tiêu

- Tái hiện các ví dụ cốt lõi của chương bằng mã có thể chạy lại.
- Kiểm tra giả định trước khi diễn giải output.
- Kết thúc bằng ít nhất một câu hỏi về điều mà kết quả **chưa** cho biết.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("error", category=FutureWarning)
warnings.filterwarnings("error", category=DeprecationWarning)
ROOT = Path.cwd()
DATA = ROOT / "data"
print("Working root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 20)


In [ ]:
df = pd.read_csv(DATA / "metromart" / "metromart_transactions.csv")
valid=df["discount_pct"].between(0,1) & df["unit_price"].notna()
work=df.loc[valid].copy()
work["net_sales"]=work["unit_price"]*work["quantity"]*(1-work["discount_pct"])
stores=pd.read_csv(DATA / "metromart" / "metromart_stores.csv")
work=work.merge(stores[["store_id","store_type","region"]],on="store_id",how="left",validate="many_to_one")

## 1. Tách – xử lý – kết hợp

In [ ]:
summary = work.groupby("store_type")["net_sales"].agg(["count","mean","median","sum"])
display(summary)

## 2. “Tốt hơn” phụ thuộc thước đo

In [ ]:
mean_rank=summary["mean"].sort_values(ascending=False)
total_rank=summary["sum"].sort_values(ascending=False)
print("Rank by mean:", list(mean_rank.index))
print("Rank by total:", list(total_rank.index))

## 3. Tỷ lệ cần mẫu số

In [ ]:
return_rate=(work["transaction_type"].eq("return").groupby(work["store_type"]).mean()).rename("return_rate")
display(return_rate)

## 4. Trung bình của trung bình

In [ ]:
group_means=work.groupby("store_type")["net_sales"].mean()
naive=group_means.mean()
overall=work["net_sales"].mean()
weighted=np.average(group_means, weights=work.groupby("store_type").size())
print("naive mean of means:", naive)
print("overall mean:", overall)
print("weighted:", weighted)

### Kiểm tra bằng chứng

Một ranking theo mean không tự động là ranking theo tổng đóng góp. Hãy nêu metric, mẫu số và số quan sát cùng nhận định.

## Thực hành

Tạo bảng `store_type × region` và kiểm tra xem nhóm đứng đầu toàn bộ dữ liệu có đứng đầu ở mọi khu vực hay không.

---
### Bạn đã sẵn sàng sang chương tiếp theo nếu có thể…

- giải thích output bằng lời;
- chỉ ra ít nhất một giả định;
- nói được kết quả chưa cho phép kết luận điều gì.

**Exit check:** Nếu mã chạy không lỗi nhưng câu trả lời trái với ý nghĩa của dữ liệu, bạn sẽ kiểm tra điều gì trước?